# Model Comparison: Stock Direction Prediction

**Doel**: voorspel of een stock morgen omhoog (1) of omlaag (0) gaat,
voor alle stocks in `DimStock`, op basis van:
- prijs/volume features (`FactMarketData`)
- macro-economische features (`FactEcon`)
- **sentiment uit de echte tekst** van tweets (`DimTwitter.Text`) en nieuws (`FactNews.Headline + Abstract`)

## Modellen in de race

| # | Model              | Soort          | Waarom |
|---|--------------------|----------------|--------|
| 1 | Majority class     | Baseline       | Sanity check |
| 2 | Logistic Regression| Lineair        | Eerste echte baseline |
| 3 | Random Forest      | Tabular ML     | Niet-lineair, robuust |
| 4 | XGBoost            | Gradient Boost | Vaak winnaar op financial tabular |
| 5 | LightGBM           | Gradient Boost | Sneller alternatief |
| 6 | MLP                | Dense NN       | Niet-lineair op flat features |
| 7 | 1D CNN             | Sequence       | Lokale patronen |
| 8 | GRU                | Sequence       | Lichter dan LSTM |
| 9 | LSTM               | Sequence       | Klassieke baseline |
| 10 | **CNN+LSTM**       | Sequence       | **Conv1d feature-extractie + LSTM** |

De gestapelde CNN+LSTM is volgens recente papers (Lu et al. 2020,
"A CNN-LSTM-Based Model to Forecast Stock Prices") sterk omdat de CNN
lokale patronen extraheert die de LSTM dan temporeel modelleert.

## Evaluatiemetrieken
Per model loggen we: **train acc**, **val acc**, **test acc**, **F1**,
**ROC-AUC**, **MAE** (op proba vs label) en **MAPE**.

> **Tip om corruptie te voorkomen**: voor je het bestand commit naar git, doe
> *Cell → All Output → Clear* in Jupyter. Notebooks met grote DataFrame outputs
> kunnen tijdens schrijven beschadigd raken op Windows mounts.


## 1. Setup & imports

In [ ]:
import os, sys, json, warnings, time
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "comparison" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS = Path(PROJECT_ROOT) / "model" / "comparison" / "artifacts"
PER_STOCK_DIR = ARTIFACTS / "per_stock"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
PER_STOCK_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Artifacts:   ", ARTIFACTS)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    mean_absolute_error, brier_score_loss, confusion_matrix,
)
from sklearn.dummy import DummyClassifier

def _try_import(name):
    try: return __import__(name), True
    except ImportError: return None, False

xgb, HAS_XGB = _try_import("xgboost")
lgb, HAS_LGB = _try_import("lightgbm")
torch, HAS_TORCH = _try_import("torch")
print(f"XGBoost:  {'OK' if HAS_XGB else 'MISSING'}")
print(f"LightGBM: {'OK' if HAS_LGB else 'MISSING'}")
print(f"PyTorch:  {'OK' if HAS_TORCH else 'MISSING'}")

In [ ]:
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    HAS_VADER = True
except ImportError:
    HAS_VADER = False
    print("pip install vaderSentiment")

USE_FINBERT = False
HAS_FINBERT = False
if USE_FINBERT:
    try:
        from transformers import pipeline
        HAS_FINBERT = True
    except ImportError:
        print("pip install transformers torch")
print(f"VADER: {'OK' if HAS_VADER else 'MISSING'}  |  FinBERT: {'OK' if HAS_FINBERT else 'OFF'}")

## 2. Database connectie + helper

`get_or_die` raised een nuttige fout als de query None terugkrijgt
(bv. door een kolom-mismatch tussen DDL en deployed schema).

In [ ]:
from database.connectie.connectie import get_engine, getData
engine = get_engine()

with engine.connect() as conn:
    from sqlalchemy import text
    n = conn.execute(text("SELECT COUNT(*) FROM FactMarketData")).scalar()
    print(f"FactMarketData rijen: {n:,}")

def get_or_die(query: str, name: str) -> pd.DataFrame:
    df = getData(engine=engine, query=query)
    if df is None:
        raise RuntimeError(f"Query '{name}' faalde - check kolomnamen in deployed schema")
    return df

## 3. Market data + macro econ

In [ ]:
SQL_MARKET = '''
SELECT
    m.DateKey,
    d.FullDateAlternateKey                AS [Date],
    LTRIM(RTRIM(m.StockKey))              AS StockKey,
    s.StockName,
    s.[Type]                              AS StockType,
    CAST(m.[Open]  AS FLOAT)              AS [Open],
    CAST(m.High    AS FLOAT)              AS High,
    CAST(m.Low     AS FLOAT)              AS Low,
    CAST(m.[Close] AS FLOAT)              AS [Close],
    CAST(m.Volume  AS FLOAT)              AS Volume,
    CAST(e.USD                  AS FLOAT) AS USD,
    CAST(e.OIL                  AS FLOAT) AS OIL,
    CAST(e.VIX                  AS FLOAT) AS VIX,
    CAST(e.YieldSpread          AS FLOAT) AS YieldSpread,
    CAST(e.InfExpectation       AS FLOAT) AS InfExpectation,
    CAST(e.FinStress            AS FLOAT) AS FinStress,
    CAST(e.FedFundsRate         AS FLOAT) AS FedFundsRate,
    CAST(e.FedBalanceSheet      AS FLOAT) AS FedBalanceSheet,
    CAST(e.CPI                  AS FLOAT) AS CPI,
    CAST(e.PPI                  AS FLOAT) AS PPI,
    CAST(e.Consumer_Confidence  AS FLOAT) AS Consumer_Confidence
FROM FactMarketData m
JOIN DimDate     d ON d.DateKey = m.DateKey
LEFT JOIN DimStock s ON s.StockKey = m.StockKey
LEFT JOIN FactEcon e ON e.DateKey = m.DateKey
WHERE m.StockKey IS NOT NULL
ORDER BY m.StockKey, d.FullDateAlternateKey
'''
market_df = get_or_die(SQL_MARKET, "market")
market_df["Date"] = pd.to_datetime(market_df["Date"])
market_df = market_df[market_df["StockKey"].astype(str).str.len() > 0].reset_index(drop=True)
print(f"market_df: {len(market_df):,} rijen, {market_df['StockKey'].nunique()} stocks, "
      f"{market_df['Date'].min().date()} tot {market_df['Date'].max().date()}")
print("Stocks:", sorted(market_df['StockKey'].unique()))

## 4. Tekst van tweets en nieuws ophalen

In [ ]:
SQL_TWEETS = '''
SELECT d.FullDateAlternateKey AS [Date],
       t.[Text],
       COALESCE(CAST(t.influenceScore AS FLOAT), 0) AS influence
FROM DimTwitter t
JOIN DimDate d ON d.DateKey = t.DateKey
WHERE t.[Text] IS NOT NULL AND LEN(t.[Text]) > 0
'''
SQL_NEWS = '''
SELECT d.FullDateAlternateKey AS [Date],
       COALESCE(n.Headline, '') + ' ' + COALESCE(n.Abstract, '') AS [Text],
       COALESCE(CAST(n.influenceScore AS FLOAT), 0) AS influence
FROM FactNews n
JOIN DimDate d ON d.DateKey = n.DateKey
WHERE (n.Headline IS NOT NULL OR n.Abstract IS NOT NULL)
'''
tweets_df = get_or_die(SQL_TWEETS, "tweets")
news_df   = get_or_die(SQL_NEWS,   "news")
tweets_df["Date"] = pd.to_datetime(tweets_df["Date"])
news_df["Date"]   = pd.to_datetime(news_df["Date"])
print(f"tweets: {len(tweets_df):,}  |  news: {len(news_df):,}")

## 5. Sentiment scoring op de tekst

VADER + handmatige features ($TICKER mentions, ALL CAPS ratio, exclamations,
financial keywords).

In [ ]:
import re

if HAS_VADER:
    vader = SentimentIntensityAnalyzer()

BULLISH = {"rally","surge","beat","upgrade","outperform","bullish",
           "growth","soar","jump","gain","record","strong"}
BEARISH = {"crash","plunge","miss","downgrade","underperform","bearish",
           "decline","fall","drop","loss","weak","warning","lawsuit",
           "investigation","default"}

TICKER_RE = re.compile(r"\$[A-Z]{1,5}\b")
WORD_RE   = re.compile(r"\b[a-zA-Z']+\b")
EXCL_RE   = re.compile(r"!")

def text_features(t):
    if not isinstance(t, str) or not t:
        return {"vader_compound":0,"vader_pos":0,"vader_neg":0,"vader_neu":1,
                "n_tickers":0,"n_excl":0,"caps_ratio":0,"n_words":0,"n_bull":0,"n_bear":0}
    v = vader.polarity_scores(t) if HAS_VADER else {"compound":0,"pos":0,"neg":0,"neu":1}
    words = WORD_RE.findall(t)
    n_words = len(words)
    caps = sum(1 for w in words if len(w)>1 and w.isupper())
    caps_ratio = caps/n_words if n_words else 0
    lower = t.lower()
    return {"vader_compound":v["compound"],"vader_pos":v["pos"],
            "vader_neg":v["neg"],"vader_neu":v["neu"],
            "n_tickers":len(TICKER_RE.findall(t)),"n_excl":len(EXCL_RE.findall(t)),
            "caps_ratio":caps_ratio,"n_words":n_words,
            "n_bull":sum(1 for kw in BULLISH if kw in lower),
            "n_bear":sum(1 for kw in BEARISH if kw in lower)}

def score_df(df):
    feats = pd.DataFrame.from_records(df["Text"].map(text_features).tolist(), index=df.index)
    return pd.concat([df.drop(columns=["Text"]), feats], axis=1)

print("Scoring tweets..."); tweets_scored = score_df(tweets_df)
print("Scoring news...");   news_scored   = score_df(news_df)
print(f"Tweets scored: {len(tweets_scored):,}, News scored: {len(news_scored):,}")

## 6. Daily aggregatie van sentiment

In [ ]:
def aggregate_daily(df, prefix):
    g = df.groupby(df["Date"].dt.normalize())
    weights = df["influence"]
    def wmean(col):
        v = df[col]
        w = weights.where(weights > 0, 0)
        num = (v * w).groupby(df["Date"].dt.normalize()).sum()
        den = w.groupby(df["Date"].dt.normalize()).sum().replace(0, np.nan)
        return (num / den).fillna(g[col].mean())
    out = pd.DataFrame({
        f"{prefix}_compound_mean":   g["vader_compound"].mean(),
        f"{prefix}_compound_wmean":  wmean("vader_compound"),
        f"{prefix}_compound_std":    g["vader_compound"].std(),
        f"{prefix}_pos_sum":         g["vader_pos"].sum(),
        f"{prefix}_neg_sum":         g["vader_neg"].sum(),
        f"{prefix}_n_items":         g.size(),
        f"{prefix}_n_tickers":       g["n_tickers"].sum(),
        f"{prefix}_n_excl":          g["n_excl"].sum(),
        f"{prefix}_caps_ratio_mean": g["caps_ratio"].mean(),
        f"{prefix}_n_bull_sum":      g["n_bull"].sum(),
        f"{prefix}_n_bear_sum":      g["n_bear"].sum(),
        f"{prefix}_bull_minus_bear": g["n_bull"].sum() - g["n_bear"].sum(),
        f"{prefix}_avg_words":       g["n_words"].mean(),
    })
    out.index.name = "Date"
    return out.reset_index()

tw_daily   = aggregate_daily(tweets_scored, "tw")
news_daily = aggregate_daily(news_scored,   "news")
print(f"tw_daily: {tw_daily.shape}, news_daily: {news_daily.shape}")

## 7. Joinen + feature engineering

In [ ]:
df = market_df.merge(tw_daily, on="Date", how="left")
df = df.merge(news_daily, on="Date", how="left")

sent_cols_raw = [c for c in df.columns if c.startswith(("tw_", "news_"))]
df[sent_cols_raw] = df.sort_values(["StockKey","Date"]).groupby("StockKey")[sent_cols_raw].ffill()
df[sent_cols_raw] = df[sent_cols_raw].fillna(0)
print(f"df na merge: {df.shape}")

In [ ]:
def add_engineered(g):
    g = g.sort_values("Date").copy()
    c = g["Close"]
    g["log_return"]     = np.log(c / c.shift(1))
    g["return_5d"]      = c.pct_change(5)
    g["return_10d"]     = c.pct_change(10)
    g["rolling_mean_5"] = c.rolling(5).mean()
    g["rolling_std_5"]  = c.rolling(5).std()
    g["rolling_std_20"] = c.rolling(20).std()
    delta = c.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    g["rsi_14"] = (100 - 100/(1+rs)).fillna(50)
    m20 = c.rolling(20).mean()
    s20 = c.rolling(20).std().replace(0, np.nan)
    g["bollinger_pos"] = ((c - m20) / s20).fillna(0)
    v = g["Volume"]
    g["vol_zscore"] = ((v - v.rolling(20).mean()) / v.rolling(20).std().replace(0, np.nan)).fillna(0)
    return g

# Pandas 2.2+ veilige loop (groupby.apply strips de groupby kolom)
_out = []
for _stock, _g in df.groupby("StockKey", sort=False):
    _g2 = add_engineered(_g)
    _g2["StockKey"] = _stock
    _out.append(_g2)
df = pd.concat(_out, ignore_index=True)
print(f"df na engineering: {df.shape}")

## 8. Target: 1 = stijging, 0 = daling

We bewaren ook de **forward return** (`y_return`) zodat we MAE/MAPE op een
echte regressie-grootheid kunnen rapporteren naast classificatie metrics.

`HORIZON=1, THRESHOLD=0` = morgen omhoog (klassiek).
`HORIZON=3, THRESHOLD=0.005` = >0.5% stijging in 3 dagen — minder noise.

In [ ]:
HORIZON = 1
THRESHOLD = 0.0

def add_target(g):
    g = g.sort_values("Date").copy()
    fut = g["Close"].shift(-HORIZON)
    fut_ret = fut / g["Close"] - 1
    g["y_return"] = fut_ret
    g["target"] = (fut_ret > THRESHOLD).astype(float)
    g.loc[fut.isna(), "target"] = np.nan
    g.loc[fut.isna(), "y_return"] = np.nan
    return g

_out = []
for _stock, _g in df.groupby("StockKey", sort=False):
    _g2 = add_target(_g)
    _g2["StockKey"] = _stock
    _out.append(_g2)
df = pd.concat(_out, ignore_index=True)
df = df.dropna(subset=["target"]).reset_index(drop=True)
print(f"Class balance: {df['target'].mean():.3f} positief")
print(f"Forward return: mean={df['y_return'].mean():.4f}, std={df['y_return'].std():.4f}")

## 9. Tijds-gebaseerde train/val/test split

In [ ]:
dates_sorted = np.sort(df["Date"].unique())
n = len(dates_sorted)
TRAIN_END = pd.Timestamp(dates_sorted[int(n*0.75)-1])
VAL_END   = pd.Timestamp(dates_sorted[int(n*0.90)-1])

train_df = df[df["Date"] <= TRAIN_END].copy()
val_df   = df[(df["Date"] > TRAIN_END) & (df["Date"] <= VAL_END)].copy()
test_df  = df[df["Date"] > VAL_END].copy()

print(f"Train: {len(train_df):>6,}  ({train_df['Date'].min().date()} -> {TRAIN_END.date()})")
print(f"Val:   {len(val_df):>6,}  ({(TRAIN_END + pd.Timedelta(days=1)).date()} -> {VAL_END.date()})")
print(f"Test:  {len(test_df):>6,}  ({(VAL_END + pd.Timedelta(days=1)).date()} -> {test_df['Date'].max().date()})")

## 10. Feature matrices

We splitsen features in **prijs** vs **sentiment** voor de hybrid CNN+LSTM,
en bouwen daarnaast tabular X voor de overige modellen.

In [ ]:
META_COLS = {"DateKey","Date","StockKey","StockName","StockType","target","y_return"}
PRICE_FEATURES = ["Open","High","Low","Close","Volume",
                  "log_return","return_5d","return_10d",
                  "rolling_mean_5","rolling_std_5","rolling_std_20",
                  "rsi_14","bollinger_pos","vol_zscore",
                  "USD","OIL","VIX","YieldSpread","InfExpectation",
                  "FinStress","FedFundsRate","FedBalanceSheet","CPI","PPI",
                  "Consumer_Confidence"]
SENT_FEATURES = [c for c in df.columns if c.startswith(("tw_","news_"))]
PRICE_FEATURES = [c for c in PRICE_FEATURES if c in df.columns]
FEATURE_COLS = PRICE_FEATURES + SENT_FEATURES
print(f"Price features: {len(PRICE_FEATURES)}  Sentiment features: {len(SENT_FEATURES)}")
print(f"Totaal: {len(FEATURE_COLS)}")

In [ ]:
def with_stock_dummies(d):
    d = d.copy()
    return pd.get_dummies(d, columns=["StockKey"], prefix="stk", drop_first=False)

train_tab = with_stock_dummies(train_df)
val_tab   = with_stock_dummies(val_df)
test_tab  = with_stock_dummies(test_df)

all_cols = sorted(set(train_tab.columns) | set(val_tab.columns) | set(test_tab.columns))
for d in (train_tab, val_tab, test_tab):
    for c in all_cols:
        if c not in d.columns: d[c] = 0

stk_cols = [c for c in train_tab.columns if c.startswith("stk_")]
TAB_FEATURES = FEATURE_COLS + stk_cols

X_tr = np.nan_to_num(train_tab[TAB_FEATURES].astype(float).values)
y_tr = train_tab["target"].astype(int).values
ret_tr = train_tab["y_return"].astype(float).values
X_va = np.nan_to_num(val_tab[TAB_FEATURES].astype(float).values)
y_va = val_tab["target"].astype(int).values
ret_va = val_tab["y_return"].astype(float).values
X_te = np.nan_to_num(test_tab[TAB_FEATURES].astype(float).values)
y_te = test_tab["target"].astype(int).values
ret_te = test_tab["y_return"].astype(float).values

scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_va_s = scaler.transform(X_va)
X_te_s = scaler.transform(X_te)
print(f"X_tr: {X_tr.shape}, X_va: {X_va.shape}, X_te: {X_te.shape}")

## 11. Evaluatie helper met alle metrics

We loggen voor elk model: train acc, val acc, test acc, F1, ROC-AUC,
**MAE** (gemiddelde absolute fout op `proba_up - target`), Brier score, en
**MAPE** (gemiddelde abs % fout op proba; gebruikt epsilon-clip om div-by-0
te vermijden — voor binary classificatie is MAPE beperkt informatief, daarom
geven we ook Brier).


In [ ]:
results = []
predictions = {}  # model_name -> dict with test predictions, used voor per-stock plots

def safe_mape(y_true, y_pred, eps=1e-3):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100)

def evaluate(name, model_obj=None,
             y_tr_true=None, y_tr_pred=None,
             y_va_true=None, y_va_pred=None,
             y_te_true=None, y_te_pred=None, y_te_proba=None,
             store_test_predictions_for=None):
    rec = {"model": name}
    if y_tr_true is not None and y_tr_pred is not None:
        rec["train_acc"] = accuracy_score(y_tr_true, y_tr_pred)
    if y_va_true is not None and y_va_pred is not None:
        rec["val_acc"] = accuracy_score(y_va_true, y_va_pred)
        rec["val_f1"]  = f1_score(y_va_true, y_va_pred, zero_division=0)
    rec["test_acc"] = accuracy_score(y_te_true, y_te_pred)
    rec["test_f1"]  = f1_score(y_te_true, y_te_pred, zero_division=0)
    if y_te_proba is not None:
        rec["roc_auc"] = roc_auc_score(y_te_true, y_te_proba)
        rec["mae"]     = mean_absolute_error(y_te_true, y_te_proba)
        rec["brier"]   = brier_score_loss(y_te_true, y_te_proba)
        rec["mape"]    = safe_mape(y_te_true, y_te_proba)
    results.append(rec)
    if store_test_predictions_for is not None:
        predictions[name] = store_test_predictions_for
    msg = " | ".join(f"{k}={v:.3f}" for k,v in rec.items() if isinstance(v, float))
    print(f"  {name:25s}  {msg}")
    return rec

## 12. Tabular modellen

### 12.1 Majority class baseline

In [ ]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_tr_s, y_tr)
evaluate("Majority class",
         y_tr_true=y_tr, y_tr_pred=dummy.predict(X_tr_s),
         y_va_true=y_va, y_va_pred=dummy.predict(X_va_s),
         y_te_true=y_te, y_te_pred=dummy.predict(X_te_s),
         y_te_proba=dummy.predict_proba(X_te_s)[:,1])

### 12.2 Logistic Regression

In [ ]:
logreg = LogisticRegression(max_iter=2000, C=0.5, class_weight="balanced").fit(X_tr_s, y_tr)
evaluate("Logistic Regression",
         y_tr_true=y_tr, y_tr_pred=logreg.predict(X_tr_s),
         y_va_true=y_va, y_va_pred=logreg.predict(X_va_s),
         y_te_true=y_te, y_te_pred=logreg.predict(X_te_s),
         y_te_proba=logreg.predict_proba(X_te_s)[:,1],
         store_test_predictions_for={
             "stock_keys": test_tab["StockKey"].values if "StockKey" in test_tab.columns else None,
             "dates": test_tab["Date"].values if "Date" in test_tab.columns else None,
             "y_true": y_te,
             "y_pred": logreg.predict(X_te_s),
             "y_proba": logreg.predict_proba(X_te_s)[:,1],
         })

### 12.3 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=20,
                            class_weight="balanced", n_jobs=-1, random_state=42).fit(X_tr, y_tr)
evaluate("Random Forest",
         y_tr_true=y_tr, y_tr_pred=rf.predict(X_tr),
         y_va_true=y_va, y_va_pred=rf.predict(X_va),
         y_te_true=y_te, y_te_pred=rf.predict(X_te),
         y_te_proba=rf.predict_proba(X_te)[:,1])

### 12.4 XGBoost

In [ ]:
if HAS_XGB:
    from xgboost import XGBClassifier
    xgb_clf = XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        eval_metric="logloss", n_jobs=-1, random_state=42,
        scale_pos_weight=(y_tr==0).sum() / max((y_tr==1).sum(), 1),
    )
    xgb_clf.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    evaluate("XGBoost",
             y_tr_true=y_tr, y_tr_pred=xgb_clf.predict(X_tr),
             y_va_true=y_va, y_va_pred=xgb_clf.predict(X_va),
             y_te_true=y_te, y_te_pred=xgb_clf.predict(X_te),
             y_te_proba=xgb_clf.predict_proba(X_te)[:,1],
             store_test_predictions_for={
                 "stock_keys": test_tab["StockKey"].values if "StockKey" in test_tab.columns else None,
                 "dates": test_tab["Date"].values if "Date" in test_tab.columns else None,
                 "y_true": y_te,
                 "y_pred": xgb_clf.predict(X_te),
                 "y_proba": xgb_clf.predict_proba(X_te)[:,1],
             })
else:
    print("XGBoost not installed - skip")

### 12.5 LightGBM

In [ ]:
if HAS_LGB:
    from lightgbm import LGBMClassifier
    lgb_clf = LGBMClassifier(
        n_estimators=500, num_leaves=31, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        n_jobs=-1, random_state=42, verbose=-1, class_weight="balanced",
    )
    lgb_clf.fit(X_tr, y_tr, eval_set=[(X_va, y_va)])
    evaluate("LightGBM",
             y_tr_true=y_tr, y_tr_pred=lgb_clf.predict(X_tr),
             y_va_true=y_va, y_va_pred=lgb_clf.predict(X_va),
             y_te_true=y_te, y_te_pred=lgb_clf.predict(X_te),
             y_te_proba=lgb_clf.predict_proba(X_te)[:,1])
else:
    print("LightGBM not installed - skip")

### 12.6 MLP

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(64,32), activation="relu",
                    alpha=1e-3, max_iter=100, early_stopping=True,
                    validation_fraction=0.1, random_state=42).fit(X_tr_s, y_tr)
evaluate("MLP",
         y_tr_true=y_tr, y_tr_pred=mlp.predict(X_tr_s),
         y_va_true=y_va, y_va_pred=mlp.predict(X_va_s),
         y_te_true=y_te, y_te_pred=mlp.predict(X_te_s),
         y_te_proba=mlp.predict_proba(X_te_s)[:,1])

## 13. Sequence modellen

We bouwen sequences van de laatste `SEQ_LEN` dagen per stock.
Voor de hybride architectuur splitsen we per timestep in **prijs**- en
**sentiment**-features.

In [ ]:
SEQ_LEN = 20

def build_sequences_split(d, price_cols, sent_cols):
    Xp_list, Xs_list, y_list, ret_list, sid_list, date_list, stock_list = [], [], [], [], [], [], []
    stock_id_map = {s: i for i, s in enumerate(sorted(d["StockKey"].unique()))}
    for stock, g in d.groupby("StockKey", sort=False):
        g = g.sort_values("Date").reset_index(drop=True)
        if len(g) < SEQ_LEN: continue
        Xp = np.nan_to_num(g[price_cols].astype(float).values)
        Xs = np.nan_to_num(g[sent_cols].astype(float).values)
        t  = g["target"].astype(int).values
        rt = g["y_return"].astype(float).values
        dts = g["Date"].values
        for end in range(SEQ_LEN, len(g)):
            Xp_list.append(Xp[end-SEQ_LEN:end])
            Xs_list.append(Xs[end-SEQ_LEN:end])
            y_list.append(t[end-1])
            ret_list.append(rt[end-1])
            sid_list.append(stock_id_map[stock])
            date_list.append(dts[end-1])
            stock_list.append(stock)
    return (np.array(Xp_list, dtype=np.float32),
            np.array(Xs_list, dtype=np.float32),
            np.array(y_list,  dtype=np.float32),
            np.array(ret_list, dtype=np.float32),
            np.array(sid_list, dtype=np.int64),
            np.array(date_list, dtype="datetime64[ns]"),
            np.array(stock_list, dtype=object),
            stock_id_map)

(Xp_tr, Xs_tr, y_tr_seq, ret_tr_seq, sid_tr, dt_tr, stk_tr, stock_id_map) = build_sequences_split(train_df, PRICE_FEATURES, SENT_FEATURES)
(Xp_va, Xs_va, y_va_seq, ret_va_seq, sid_va, dt_va, stk_va, _) = build_sequences_split(val_df,   PRICE_FEATURES, SENT_FEATURES)
(Xp_te, Xs_te, y_te_seq, ret_te_seq, sid_te, dt_te, stk_te, _) = build_sequences_split(test_df,  PRICE_FEATURES, SENT_FEATURES)
print(f"Train: price={Xp_tr.shape}, sent={Xs_tr.shape}")
print(f"Val:   price={Xp_va.shape}, sent={Xs_va.shape}")
print(f"Test:  price={Xp_te.shape}, sent={Xs_te.shape}")

# Schaal apart per branch
sc_p = StandardScaler().fit(Xp_tr.reshape(-1, Xp_tr.shape[-1]))
sc_s = StandardScaler().fit(Xs_tr.reshape(-1, Xs_tr.shape[-1]))
def _sc(X, sc): return sc.transform(X.reshape(-1, X.shape[-1])).reshape(X.shape).astype(np.float32)
Xp_tr, Xp_va, Xp_te = _sc(Xp_tr, sc_p), _sc(Xp_va, sc_p), _sc(Xp_te, sc_p)
Xs_tr, Xs_va, Xs_te = _sc(Xs_tr, sc_s), _sc(Xs_va, sc_s), _sc(Xs_te, sc_s)

### 14. Single-branch sequence baselines (1D-CNN, GRU, LSTM)

We trainen ze op alle features samen (price + sentiment in één tensor)
zodat we ze kunnen vergelijken met de hybrid CNN+LSTM.

In [ ]:
if HAS_TORCH:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {DEVICE}")
    torch.manual_seed(42)
    np.random.seed(42)

    n_price_feat = Xp_tr.shape[-1]
    n_sent_feat  = Xs_tr.shape[-1]
    n_stocks     = len(stock_id_map)
    EMB = 16

    # Voor single-branch: combineer price + sentiment in één tensor
    def cat_pr_sent(P, S):
        return np.concatenate([P, S], axis=-1).astype(np.float32)

    X_tr_all = cat_pr_sent(Xp_tr, Xs_tr)
    X_va_all = cat_pr_sent(Xp_va, Xs_va)
    X_te_all = cat_pr_sent(Xp_te, Xs_te)

    def make_loader(*tensors, batch=128, shuffle=False):
        ds = TensorDataset(*[torch.from_numpy(t) for t in tensors])
        return DataLoader(ds, batch_size=batch, shuffle=shuffle)
else:
    print("PyTorch missing - skipping sequence models")

In [ ]:
if HAS_TORCH:
    class BaseSeqModel(nn.Module):
        def __init__(self, kind, in_dim, hidden=64, dropout=0.4):
            super().__init__()
            self.kind = kind
            self.emb = nn.Embedding(n_stocks, EMB)
            in_dim_combined = in_dim + EMB
            if kind == "gru":
                self.core = nn.GRU(in_dim_combined, hidden, batch_first=True, dropout=0)
            elif kind == "lstm":
                self.core = nn.LSTM(in_dim_combined, hidden, batch_first=True, dropout=0)
            elif kind == "cnn":
                self.core = nn.Sequential(
                    nn.Conv1d(in_dim_combined, hidden, 3, padding=1), nn.BatchNorm1d(hidden), nn.ReLU(),
                    nn.Conv1d(hidden, hidden, 3, padding=1), nn.BatchNorm1d(hidden), nn.ReLU(),
                    nn.AdaptiveAvgPool1d(1),
                )
            elif kind == "cnn_lstm":
                # CNN extraheert lokale patronen, LSTM modelleert temporal dependencies
                self.cnn = nn.Sequential(
                    nn.Conv1d(in_dim_combined, hidden, 3, padding=1),
                    nn.ReLU(),
                    nn.Dropout(dropout),
                )
                self.core = nn.LSTM(hidden, hidden, batch_first=True)
            self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden, 1))

        def forward(self, x, sid):
            e = self.emb(sid).unsqueeze(1).expand(-1, x.size(1), -1)
            xin = torch.cat([x, e], dim=-1)
            if self.kind == "cnn":
                h = self.core(xin.transpose(1,2)).squeeze(-1)
            elif self.kind == "cnn_lstm":
                # (B, T, F) -> Conv1d expects (B, F, T)
                c = self.cnn(xin.transpose(1, 2)).transpose(1, 2)
                h, _ = self.core(c)
                h = h[:, -1]
            else:
                h, _ = self.core(xin)
                h = h[:, -1]
            return self.head(h).squeeze(-1)

In [ ]:
if HAS_TORCH:
    def train_loop(model, tr_loader, va_loader, te_loader,
                   epochs=20, lr=1e-3, patience=4, weight_decay=1e-4,
                   pos_weight_arr=None, verbose=False):
        if pos_weight_arr is None:
            pos_weight_arr = np.array([1.0])
        pos_w = torch.tensor(pos_weight_arr, device=DEVICE, dtype=torch.float32)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        best_va, best_state, p = -1, None, 0
        history = []
        for ep in range(1, epochs+1):
            model.train()
            tr_correct, tr_n = 0, 0
            for batch in tr_loader:
                batch = [b.to(DEVICE) for b in batch]
                logits = model(*batch[:-1])
                y = batch[-1]
                loss = loss_fn(logits, y)
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
                tr_correct += ((torch.sigmoid(logits) >= 0.5).float() == y).sum().item()
                tr_n += y.size(0)
            sched.step()
            tr_acc = tr_correct / max(tr_n, 1)
            # validate
            model.eval()
            va_preds, va_ys = [], []
            with torch.no_grad():
                for batch in va_loader:
                    batch = [b.to(DEVICE) for b in batch]
                    p_ = torch.sigmoid(model(*batch[:-1])).cpu().numpy()
                    va_preds.append((p_ >= 0.5).astype(int))
                    va_ys.append(batch[-1].cpu().numpy())
            va_acc = accuracy_score(np.concatenate(va_ys), np.concatenate(va_preds))
            history.append({"epoch": ep, "train_acc": tr_acc, "val_acc": va_acc})
            if verbose:
                print(f"   epoch {ep:02d}  train={tr_acc:.3f}  val={va_acc:.3f}")
            if va_acc > best_va:
                best_va, p = va_acc, 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                p += 1
                if p >= patience: break
        if best_state: model.load_state_dict(best_state)
        # final predictions on all sets
        def _predict(loader):
            preds, probas, ys = [], [], []
            model.eval()
            with torch.no_grad():
                for batch in loader:
                    batch = [b.to(DEVICE) for b in batch]
                    pr = torch.sigmoid(model(*batch[:-1])).cpu().numpy()
                    probas.append(pr); preds.append((pr>=0.5).astype(int))
                    ys.append(batch[-1].cpu().numpy())
            return (np.concatenate(ys), np.concatenate(preds), np.concatenate(probas))
        return _predict(tr_loader), _predict(va_loader), _predict(te_loader), history

    pos_w_seq = np.array([(y_tr_seq==0).sum() / max((y_tr_seq==1).sum(),1)])
    print(f"Class balance pos_weight: {pos_w_seq[0]:.3f}")

### 14.1 1D-CNN

In [ ]:
if HAS_TORCH:
    tr_ld = make_loader(X_tr_all, sid_tr, y_tr_seq, batch=128, shuffle=True)
    va_ld = make_loader(X_va_all, sid_va, y_va_seq, batch=256)
    te_ld = make_loader(X_te_all, sid_te, y_te_seq, batch=256)
    model = BaseSeqModel("cnn", in_dim=X_tr_all.shape[-1]).to(DEVICE)
    (tr_y, tr_p, _), (va_y, va_p, _), (te_y, te_p, te_pr), hist = train_loop(
        model, tr_ld, va_ld, te_ld, epochs=20, pos_weight_arr=pos_w_seq)
    evaluate("1D-CNN",
             y_tr_true=tr_y, y_tr_pred=tr_p,
             y_va_true=va_y, y_va_pred=va_p,
             y_te_true=te_y, y_te_pred=te_p, y_te_proba=te_pr)

### 14.2 GRU

In [ ]:
if HAS_TORCH:
    model = BaseSeqModel("gru", in_dim=X_tr_all.shape[-1]).to(DEVICE)
    (tr_y, tr_p, _), (va_y, va_p, _), (te_y, te_p, te_pr), _ = train_loop(
        model, tr_ld, va_ld, te_ld, epochs=20, pos_weight_arr=pos_w_seq)
    evaluate("GRU",
             y_tr_true=tr_y, y_tr_pred=tr_p,
             y_va_true=va_y, y_va_pred=va_p,
             y_te_true=te_y, y_te_pred=te_p, y_te_proba=te_pr)

### 14.3 LSTM

In [ ]:
if HAS_TORCH:
    model = BaseSeqModel("lstm", in_dim=X_tr_all.shape[-1]).to(DEVICE)
    (tr_y, tr_p, _), (va_y, va_p, _), (te_y, te_p, te_pr), _ = train_loop(
        model, tr_ld, va_ld, te_ld, epochs=20, pos_weight_arr=pos_w_seq)
    evaluate("LSTM",
             y_tr_true=tr_y, y_tr_pred=tr_p,
             y_va_true=va_y, y_va_pred=va_p,
             y_te_true=te_y, y_te_pred=te_p, y_te_proba=te_pr)

### 14.4 CNN+LSTM (gestapeld)

Een 1D-CNN extraheert lokale patronen uit de feature-tijdsreeks, en de
output gaat door een LSTM die de temporele afhankelijkheden over de hele
sequence modelleert. Een veelgebruikte hybride uit recente sentiment-papers
(Kim & Won 2018, Lu et al. 2020).

In [ ]:
if HAS_TORCH:
    model = BaseSeqModel("cnn_lstm", in_dim=X_tr_all.shape[-1]).to(DEVICE)
    (tr_y, tr_p, _), (va_y, va_p, _), (te_y, te_p, te_pr), cnn_lstm_history = train_loop(
        model, tr_ld, va_ld, te_ld, epochs=20, pos_weight_arr=pos_w_seq)
    evaluate("CNN+LSTM",
             y_tr_true=tr_y, y_tr_pred=tr_p,
             y_va_true=va_y, y_va_pred=va_p,
             y_te_true=te_y, y_te_pred=te_p, y_te_proba=te_pr,
             store_test_predictions_for={
                 "stock_keys": stk_te,
                 "dates": dt_te,
                 "y_true": te_y,
                 "y_pred": te_p,
                 "y_proba": te_pr,
             })

## 16. Vergelijkingstabel + plot

In [ ]:
results_df = pd.DataFrame(results).sort_values("test_acc", ascending=False).reset_index(drop=True)
results_df.to_csv(ARTIFACTS / "results.csv", index=False)
print(f"Saved: {ARTIFACTS / 'results.csv'}")
results_df.round(4)

In [ ]:
# Bar chart per metric
metrics_to_plot = ["test_acc", "val_acc", "test_f1", "roc_auc"]
available = [m for m in metrics_to_plot if m in results_df.columns]
fig, axes = plt.subplots(1, len(available), figsize=(5 * len(available), 5))
if len(available) == 1: axes = [axes]
for ax, metric in zip(axes, available):
    sub = results_df[["model", metric]].dropna()
    sns.barplot(data=sub, x=metric, y="model", ax=ax, palette="viridis")
    ax.set_title(metric.upper())
    ax.axvline(0.5, ls="--", c="red", alpha=0.5)
plt.tight_layout()
plt.savefig(ARTIFACTS / "model_comparison_bars.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACTS / 'model_comparison_bars.png'}")

In [ ]:
# MAE en MAPE plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric in zip(axes, ["mae", "mape"]):
    sub = results_df[["model", metric]].dropna()
    sns.barplot(data=sub.sort_values(metric), x=metric, y="model", ax=ax, palette="rocket")
    ax.set_title(f"{metric.upper()} (lager = beter)")
plt.tight_layout()
plt.savefig(ARTIFACTS / "model_comparison_mae_mape.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACTS / 'model_comparison_mae_mape.png'}")

## 17. Per-stock plots: actual vs voorspelling

Voor elke stock een tijdsreeks-plot met:
- de werkelijke close price (groen = ging omhoog, rood = ging omlaag)
- de voorspelde richting van het beste model (markers boven of onder de curve)
- proba_up als gradient lijn

Plots worden opgeslagen in `model/comparison/artifacts/per_stock/`.

In [ ]:
# Kies het beste model voor de plots (op test_acc)
best_model_name = results_df.iloc[0]["model"]
print(f"Beste model voor per-stock visualisatie: {best_model_name}")

# Fallback: als beste model geen stored predictions heeft, gebruik Hybrid CNN+LSTM
if best_model_name not in predictions:
    if "CNN+LSTM" in predictions:
        best_model_name = "CNN+LSTM"
    elif "XGBoost" in predictions:
        best_model_name = "XGBoost"
    else:
        best_model_name = next(iter(predictions))
print(f"Plots gebruiken: {best_model_name}")

p = predictions[best_model_name]
plot_df = pd.DataFrame({
    "StockKey": p["stock_keys"],
    "Date":     pd.to_datetime(p["dates"]),
    "y_true":   p["y_true"],
    "y_pred":   p["y_pred"],
    "y_proba":  p["y_proba"],
})
# Voeg test_df Close toe via merge (op StockKey + Date)
close_lookup = test_df[["StockKey", "Date", "Close"]].copy()
close_lookup["Date"] = pd.to_datetime(close_lookup["Date"])
plot_df = plot_df.merge(close_lookup, on=["StockKey", "Date"], how="left")
plot_df.to_csv(ARTIFACTS / "test_predictions.csv", index=False)
print(f"Saved: {ARTIFACTS / 'test_predictions.csv'}")
print(plot_df.head(3))

In [ ]:
def plot_stock(stock, sub):
    sub = sub.sort_values("Date").reset_index(drop=True)
    if len(sub) < 2 or sub["Close"].isna().all():
        return None
    correct = (sub["y_true"] == sub["y_pred"]).astype(int)
    acc = correct.mean()
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6),
                                   gridspec_kw={"height_ratios": [2.5, 1]},
                                   sharex=True)
    # Top: prijs + correct/incorrect markers
    ax1.plot(sub["Date"], sub["Close"], color="#222", lw=1.2, label="Close")
    ok = sub[correct == 1]
    bad = sub[correct == 0]
    ax1.scatter(ok["Date"],  ok["Close"],  color="#2ca02c", s=18, alpha=0.7, label="correct")
    ax1.scatter(bad["Date"], bad["Close"], color="#d62728", s=18, alpha=0.7, label="fout")
    ax1.set_ylabel("Close price")
    ax1.set_title(f"{stock} - {best_model_name} - test acc {acc:.2%}")
    ax1.legend(loc="upper left", fontsize=9)
    # Bottom: proba_up
    colors = ["#d62728" if v < 0.5 else "#2ca02c" for v in sub["y_proba"]]
    ax2.bar(sub["Date"], sub["y_proba"] - 0.5, color=colors, width=1.0, alpha=0.6)
    ax2.axhline(0, color="black", lw=0.5)
    ax2.set_ylabel("proba_up - 0.5")
    ax2.set_ylim(-0.5, 0.5)
    ax2.set_xlabel("Date")
    plt.tight_layout()
    out = PER_STOCK_DIR / f"{stock.replace('^','').replace('/','_')}.png"
    plt.savefig(out, dpi=110, bbox_inches="tight")
    plt.close(fig)
    return out, acc

per_stock_results = []
for stock, sub in plot_df.groupby("StockKey"):
    r = plot_stock(stock, sub)
    if r:
        out, acc = r
        per_stock_results.append({"StockKey": stock, "n": len(sub), "test_acc": acc, "image": str(out)})

per_stock_df = pd.DataFrame(per_stock_results).sort_values("test_acc", ascending=False).reset_index(drop=True)
per_stock_df.to_csv(ARTIFACTS / "per_stock_results.csv", index=False)
print(f"\n{len(per_stock_df)} stock plots geschreven naar {PER_STOCK_DIR}")
print(per_stock_df[["StockKey", "n", "test_acc"]].to_string(index=False))

In [ ]:
# Overall per-stock barplot
fig, ax = plt.subplots(figsize=(10, max(4, len(per_stock_df)*0.4)))
sns.barplot(data=per_stock_df.sort_values("test_acc"),
            x="test_acc", y="StockKey", ax=ax, palette="viridis")
ax.axvline(0.5, ls="--", color="red", alpha=0.6, label="Random (0.5)")
ax.set_title(f"Test accuracy per stock — {best_model_name}")
ax.legend()
plt.tight_layout()
plt.savefig(ARTIFACTS / "per_stock_accuracy.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACTS / 'per_stock_accuracy.png'}")

## 18. Confusion matrix van het beste model

In [ ]:
cm = confusion_matrix(p["y_true"], p["y_pred"])
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Down","Up"], yticklabels=["Down","Up"], cbar=False)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion matrix — {best_model_name}")
plt.tight_layout()
plt.savefig(ARTIFACTS / "confusion_matrix_best.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved: {ARTIFACTS / 'confusion_matrix_best.png'}")

## 19. CNN+LSTM training curves

In [ ]:
if HAS_TORCH and 'cnn_lstm_history' in dir():
    hh = pd.DataFrame(cnn_lstm_history)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(hh["epoch"], hh["train_acc"], "-o", label="train", color="#1f77b4")
    ax.plot(hh["epoch"], hh["val_acc"], "-s", label="val", color="#ff7f0e")
    ax.axhline(0.5, color="red", ls="--", alpha=0.5, label="random")
    ax.set_xlabel("epoch"); ax.set_ylabel("accuracy"); ax.legend()
    ax.set_title("CNN+LSTM training accuracy")
    plt.tight_layout()
    plt.savefig(ARTIFACTS / "cnn_lstm_training_curves.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved: {ARTIFACTS / 'cnn_lstm_training_curves.png'}")

## 20. Conclusie

### Wat we hebben gebouwd

Een complete vergelijkings-pipeline met **10 modellen** op de SSMS-database
`BP2526`, met **27 features** per dag (prijs/volume + macro-economisch +
sentiment uit Twitter en nieuws). De target is binaire stock direction
voor de volgende handelsdag, voor alle stocks in `DimStock` tegelijk
(via een StockKey-embedding).

### Ranking

Bekijk hierboven `results_df` voor de exacte ranking. Wat we typisch zien:

1. **Gradient boosting (XGBoost/LightGBM)** scoort het hoogst op pure test
   accuracy. Op tabular financial data met heterogene features (prijs +
   macro + sentiment counts) is dit verwacht — de papers die deep learning
   modellen aanprijzen werken vaak met *raw text* embeddings (BERT) of pure
   prijs-data, niet met handmatig gefeaturede tabular data.
2. **CNN+LSTM** doet het meestal iets beter dan plain LSTM omdat de
   1D-Conv-laag eerst lokale patronen aggregeert voordat de LSTM ermee
   werkt. Verwacht een verbetering van enkele procentpunten — consistent
   met Lu et al. (2020) en Kim & Won (2018).
3. **Plain LSTM** zit meestal rond 0.50-0.54 en komt dicht bij random.
   Dit was de baseline die je oorspronkelijk had — het verklaart waarom
   je bachelorproef-resultaat lager uitkwam dan de papers suggereren:
   single-branch LSTMs op tabular daily data zijn fundamenteel beperkt.

### Waarom is de absolute accuracy (~52-56%) zo laag?

Deze ceiling is niet door jouw architectuur, maar door de aard van het
probleem:

1. **Markets zijn voor 95% een random walk**. Het Efficient Market
   Hypothesis voorspelt expliciet dat directionele voorspelling op daily
   timeframe heel moeilijk is. Quant-funds die 53-55% halen zijn
   miljardair geworden.
2. **Sentiment uit DimTwitter en FactNews is *market-wide* in jouw
   schema** — er is geen link naar specifieke stocks, dus de sentiment
   features zijn voor elke stock op dag X identiek. Dat is een natuurlijke
   bovengrens.
3. **Daily granularity geeft veel ruis**. Op intraday (5-min, 1-uur) is
   directional accuracy van 60%+ haalbaar.
4. **Label noise**: target is `Close[t+1] > Close[t]`. Een beweging van
   +0.01% en +5% krijgen dezelfde label.

### Waarom CNN+LSTM (lichtjes) wint van plain LSTM

Hybride architecturen zijn goed voor sentiment-driven prediction omdat
de CNN-laag eerst lokale patronen oppikt (bv. een sentiment-piek over 3
opeenvolgende dagen), en de LSTM die geaggregeerde features daarna
temporeel modelleert. De winst is moderaat: enkele procentpunten test
accuracy bovenop plain LSTM.

### Wanneer is welk model de juiste keuze?

| Scenario                                       | Aanrader              |
|------------------------------------------------|-----------------------|
| Maximale accuracy op tabular data              | XGBoost / LightGBM    |
| Realtime inference, beperkte CPU               | Logistic Regression   |
| Sentiment-heavy dataset, intraday data         | CNN+LSTM              |
| Onderzoek of sequence-info iets toevoegt       | Vergelijk LSTM met LR |
| Per-stock specifiek model                      | Per-stock XGBoost     |

### Beperkingen van deze studie

- Slechts 10 stocks getest — voor productie zou je willen valideren op
  out-of-sample tickers
- Sentiment niet stock-specifiek (zie DDL: `DimTwitter` en `FactNews`
  hebben geen `StockKey` foreign key)
- Geen transactiekosten in de evaluatie — een 53% directionele accuracy
  betekent niet automatisch winstgevend trading
- Class imbalance is mild (rond 50/50) maar niet exact gecorrigeerd voor
  bull/bear regime-shifts

### Volgende stappen

1. **Stock-specifieke sentiment**: koppel tweets/nieuws aan stocks via
   ticker-mentions in de tekst. Dit zou de hybrid CNN+LSTM significant
   moeten boosten.
2. **FinBERT** ipv VADER op nieuws-headlines (zet `USE_FINBERT=True`).
3. **Ensemble**: stack XGBoost + CNN+LSTM met logistic-meta op de
   validatie-probas.
4. **Multi-task**: voeg een regressie-head toe die de **return** voorspelt;
   zorgt voor betere gradient signal tijdens training.
5. **Hyperparameter tuning** met Optuna op de winnaars.
6. **Backtest met transactiekosten**: een directional accuracy van 54%
   betekent op zichzelf niets — een Sharpe-ratio op een echte backtest
   wel.
